In [56]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [57]:
from pyspark.sql import SparkSession
# For Linear Interpolation
from pyspark.sql import Window
import pyspark.sql.functions as F

from pyspark.sql.functions import dayofweek, count
from pyspark.sql.functions import explode, sequence, to_date, min, max, lit

##### Data Ingestion


Initialize a Spark session

In [58]:
spark = SparkSession.builder.appName("SMS_Spam_Detection").getOrCreate()

Read the dataset into a PySpark 

In [59]:
df = spark.read.csv('all_stocks_5yr.csv', header=True, inferSchema=True)
df.show(5)

+----------+-----+-----+-----+-----+--------+----+
|      date| open| high|  low|close|  volume|Name|
+----------+-----+-----+-----+-----+--------+----+
|2013-02-08|15.07|15.12|14.63|14.75| 8407500| AAL|
|2013-02-11|14.89|15.01|14.26|14.46| 8882000| AAL|
|2013-02-12|14.45|14.51| 14.1|14.27| 8126000| AAL|
|2013-02-13| 14.3|14.94|14.25|14.66|10259500| AAL|
|2013-02-14|14.94|14.96|13.16|13.99|31879900| AAL|
+----------+-----+-----+-----+-----+--------+----+
only showing top 5 rows


Display schema and sample rows

In [60]:
print("schema:")
df.printSchema()
print("sample rows:")
df.show(5)

schema:
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: integer (nullable = true)
 |-- Name: string (nullable = true)

sample rows:
+----------+-----+-----+-----+-----+--------+----+
|      date| open| high|  low|close|  volume|Name|
+----------+-----+-----+-----+-----+--------+----+
|2013-02-08|15.07|15.12|14.63|14.75| 8407500| AAL|
|2013-02-11|14.89|15.01|14.26|14.46| 8882000| AAL|
|2013-02-12|14.45|14.51| 14.1|14.27| 8126000| AAL|
|2013-02-13| 14.3|14.94|14.25|14.66|10259500| AAL|
|2013-02-14|14.94|14.96|13.16|13.99|31879900| AAL|
+----------+-----+-----+-----+-----+--------+----+
only showing top 5 rows


##### Data Cleaning Preprocessing

Handling Missing Values

In [61]:
df.createOrReplaceTempView("stocks") # Create a temporary view for SQL queries

In [62]:
null_analysis_sql = spark.sql("""
    SELECT 
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN open IS NULL THEN 1 ELSE 0 END) AS null_open,
        SUM(CASE WHEN high IS NULL THEN 1 ELSE 0 END) AS null_high,
        SUM(CASE WHEN low IS NULL THEN 1 ELSE 0 END) AS null_low,
        SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) AS null_close,
        SUM(CASE WHEN volume IS NULL THEN 1 ELSE 0 END) AS null_volume,
        SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_Name,
        COUNT(*) AS total_rows
    FROM stocks
""")
null_analysis_sql.show()

+---------+---------+---------+--------+----------+-----------+---------+----------+
|null_date|null_open|null_high|null_low|null_close|null_volume|null_Name|total_rows|
+---------+---------+---------+--------+----------+-----------+---------+----------+
|        0|       11|        8|       8|         0|          0|        0|    619040|
+---------+---------+---------+--------+----------+-----------+---------+----------+



Fill Missing Values by Forward Fill

In [63]:
window_spec = Window.partitionBy("Name").orderBy("date")

def fill_nulls(df, column_name):
    # Forward Fill (from the previous value)
    df = df.withColumn(
        column_name, 
        F.last(column_name, True).over(window_spec.rowsBetween(Window.unboundedPreceding, 0))
    )
    return df

for col_name in ["high", "low"]:
    df = fill_nulls(df, col_name)

df = df.withColumn(
    "open", 
    F.coalesce(F.col("open"), F.lag("close", 1).over(window_spec))
)

df = spark.sql("""
    SELECT * FROM stocks 
    WHERE date IS NOT NULL 
      AND open IS NOT NULL 
      AND high IS NOT NULL 
      AND low IS NOT NULL 
      AND close IS NOT NULL 
      AND volume IS NOT NULL 
      AND Name IS NOT NULL
""")

df.createOrReplaceTempView("stocks")

spark.sql("SELECT COUNT(*) FROM stocks WHERE open IS NULL").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



Handling Duplicates

In [64]:
duplicate_check = spark.sql("""
    SELECT date, Name, COUNT(*) as occurrence
    FROM stocks
    GROUP BY date, Name
    HAVING COUNT(*) > 1
""")

duplicate_check.show()
print(f"Number of duplicate: {duplicate_check.count()}")

+----+----+----------+
|date|Name|occurrence|
+----+----+----------+
+----+----+----------+



Number of duplicate: 0


Handling Outliers

In [65]:
quantiles = df.approxQuantile("volume", [0.25, 0.75], 0.01)

# IQR = Q3 -Q1
IQR = quantiles[1] - quantiles[0]

# Upper bound = Q3 + 1.5 * IQR
upper_bound = quantiles[1] + 1.5 * IQR
# Lower bound = Q1 - 1.5 * IQR
lower_bound = quantiles[0] - 1.5 * IQR

outliers_df = df.filter((F.col("volume") > upper_bound) | (F.col("volume") < lower_bound))
outliers_count = outliers_df.count()
total_count = df.count()

print(f"Total Rows: {total_count}")
print(f"Lower Bound: {lower_bound}")
print(f"Upper Bound: {upper_bound}")
print(f"Number of Outliers detected: {outliers_count}")
print(f"Percentage of Outliers: {(outliers_count / total_count) * 100:.2f}%")

Total Rows: 619029
Lower Bound: -3669163.0
Upper Bound: 8935141.0
Number of Outliers detected: 61100
Percentage of Outliers: 9.87%


Ensuring Dataset Contains Only Stock Market Business Days (Mon–Fri)

In [66]:
df_check = df.withColumn("day_num", dayofweek("date")) # 1 = Sunday, 2 = Monday, ..., 7 = Saturday

weekend_analysis = df_check.groupBy("day_num").agg(count("*").alias("record_count")).orderBy("day_num")

weekend_analysis.show()

+-------+------------+
|day_num|record_count|
+-------+------------+
|      2|      115998|
|      3|      127827|
|      4|      127376|
|      5|      123922|
|      6|      123906|
+-------+------------+

